# Runtime Safety Showcase: Stale Data, Kill Switch, Reconciliation, Health

**Docker image**: `ml4t`

**Section Reference**: 25.7 (Operational readiness)

**Implementation Skills**:
- `ml4t.live.safety`: enforced `max_data_staleness_seconds` and `max_daily_loss`
- `ml4t.live.safety`: persisted `RiskState` and `SafeBroker.connect()` startup reconciliation
- `ml4t.live.engine`: `LiveEngine.runtime_status()` and the `ok` / `waiting_for_data` /
  `feed_silent` / `idle_market_closed` / `broker_disconnected` / `stopped` health states
- `ml4t-live` CLI as an out-of-process operator surface

**Why This Notebook Exists**

`10_safety_risk_demo` walks through the configurable risk surface, the settings the strategy
author exposes to the operator. This notebook exercises the runtime trust contract: what
happens *under failure* in the kinds of scenarios a live deployment must survive. A pre-flight
checklist is only useful if the controls actually fire when the conditions they protect
against arise. Here each control is driven into the failure mode it was designed for.

**Learning Objectives**
- Watch `SafeBroker` reject orders against a stale `MarketSnapshot` and against a daily-loss
  breach, and confirm the kill switch latches across `SafeBroker` reconstruction.
- Inspect a non-clean `reconciliation_report` from a deliberately divergent persisted state
  file, then resolve it and reconnect to a clean report.
- Read `LiveEngine.runtime_status()` and observe the engine transition through the health
  states named in §25.7.
- Use the `ml4t-live` CLI to inspect the persisted state file out of process.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 執行階段安全示範：過時資料、Kill Switch、對帳與健康狀態

**Docker 映像檔**：`ml4t`

**章節參考**：25.7（營運準備）

**實作技能**：
- `ml4t.live.safety`：強制執行 `max_data_staleness_seconds` 與 `max_daily_loss`
- `ml4t.live.safety`：持久化的 `RiskState` 與 `SafeBroker.connect()` 啟動時對帳
- `ml4t.live.engine`：`LiveEngine.runtime_status()`，以及 `ok`／`waiting_for_data`／`feed_silent`／`idle_market_closed`／`broker_disconnected`／`stopped` 健康狀態
- `ml4t-live` CLI，作為程序外的操作人員介面

**本 Notebook 的目的**

`10_safety_risk_demo` 逐一說明可設定的風險介面，也就是策略作者提供給操作人員的設定。本 Notebook 則測試執行階段的信任契約：實盤部署必須承受的各種*失敗情境*下會發生什麼事。啟動前檢查清單只有在受防護情況出現時控制措施確實觸發，才有實際用途。此處讓每項控制措施進入其設計所針對的失敗模式。

**學習目標**
- 觀察 `SafeBroker` 如何拒絕依據過時 `MarketSnapshot` 及超過每日損失限額的訂單，並確認 Kill Switch 在重建 `SafeBroker` 後仍維持鎖定。
- 檢視刻意讓持久化狀態檔產生差異後得到的不一致 `reconciliation_report`，再解決差異並重新連線，取得一致報告。
- 讀取 `LiveEngine.runtime_status()`，觀察引擎如何在 §25.7 所列的健康狀態之間轉換。
- 使用 `ml4t-live` CLI，在程序外檢查持久化狀態檔。
<!-- ML4T_ZH_END c0000-p000 -->

**Prerequisites**
- `ml4t-live` installed at the version `pyproject.toml` pins (`uv sync` from repo root).
- Read §25.7 for the operational framing; `10_safety_risk_demo` for the configurable surface.
- No broker credentials and no exchange access: every demo runs against a synthetic broker.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**先備條件**
- 安裝 `ml4t-live`，版本須為 `pyproject.toml` 固定的版本（從儲存庫根目錄執行 `uv sync`）。
- 閱讀 §25.7 的營運背景，以及 `10_safety_risk_demo` 的可設定介面。
- 不需要券商憑證或交易所存取權限：每個示範都在合成券商上執行。
<!-- ML4T_ZH_END c0000-p001 -->


## Setup

Imports, logging, and a minimal asynchronous broker that satisfies the protocol `SafeBroker`
expects. The synthetic broker is deliberately stateful so each demo can shape its position,
order, and account-value snapshots without involving a live venue.

<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 設定

匯入模組、設定日誌，以及建立符合 `SafeBroker` 所預期協定的簡易非同步券商。合成券商刻意保留狀態，讓每個示範都能塑造其部位、訂單與帳戶價值快照，而不涉及即時平台。
<!-- ML4T_ZH_END c0001-p000 -->


In [1]:
"""Runtime Safety Showcase: stale-data, daily-loss, reconciliation, and engine health states."""

import asyncio
import json
import logging
import os
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import polars as pl
from ml4t.backtest.strategy import Strategy
from ml4t.backtest.types import Order, OrderSide, OrderStatus, OrderType, Position
from ml4t.live import LiveRiskConfig, RiskLimitError, RiskState, SafeBroker
from ml4t.live.engine import LiveEngine

from utils.paths import get_output_dir

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)


def run_demo(awaitable):
    """Run an async demo outside the notebook kernel's active event loop."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(lambda: asyncio.run(awaitable)).result()

In [2]:
# Production defaults; papermill may inject overrides
STALE_DATA_WAIT_SECONDS = 1.5  # how long to sleep before retrying with stale data
KILL_SWITCH_LOSS_USD = 1_000.0  # synthetic equity drawdown driving the trip
HEALTH_OBSERVATION_SECONDS = 6  # total wall time spent observing engine health states
STATE_DIR = get_output_dir(25, "runtime_safety_showcase") / "temporary_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

### A Synthetic Broker

The query surface exposes only the state `SafeBroker` needs. Order submission follows in a
separate cell so readers can distinguish broker observation from broker mutation.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
### 合成券商

查詢介面只暴露 `SafeBroker` 所需的狀態。送單功能放在另一個儲存格，讓讀者區分券商狀態觀測與券商狀態變動。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
class DemoBrokerQueries:
    """Connection and read-only methods required by `SafeBroker`."""

    async def connect(self) -> None:
        self._connected = True

    async def disconnect(self) -> None:
        self._connected = False

    async def is_connected_async(self) -> bool:
        return self._connected

    @property
    def execution_capabilities(self):
        return frozenset()

    def assert_paper_trading(self) -> None:
        return None

    def get_position(self, asset: str) -> Position | None:
        return self.positions.get(asset)

    async def get_position_async(self, asset: str) -> Position | None:
        return self.positions.get(asset)

    async def get_positions_async(self) -> dict[str, Position]:
        return dict(self.positions)

    async def get_pending_orders_async(self) -> list[Order]:
        return list(self.pending_orders)

    async def get_account_value_async(self) -> float:
        return self.account_value

    async def get_cash_async(self) -> float:
        return self.account_value

The mutable broker appends synthetic orders locally and never opens an external connection.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
可變動狀態的券商只會在本機附加合成訂單，絕不建立外部連線。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
class DemoBroker(DemoBrokerQueries):
    def __init__(
        self,
        *,
        positions: dict[str, Position] | None = None,
        pending_orders: list[Order] | None = None,
        account_value: float = 100_000.0,
    ) -> None:
        self._connected = False
        self.positions = dict(positions or {})
        self.pending_orders = list(pending_orders or [])
        self.account_value = float(account_value)
        self._order_counter = 0

    async def submit_order_async(
        self,
        asset: str,
        quantity: float,
        side: OrderSide | None = None,
        order_type: OrderType = OrderType.MARKET,
        limit_price: float | None = None,
        stop_price: float | None = None,
        **_: Any,
    ) -> Order:
        if side is None:
            side = OrderSide.BUY if quantity > 0 else OrderSide.SELL
            quantity = abs(quantity)
        self._order_counter += 1
        order = Order(
            asset=asset,
            side=side,
            quantity=quantity,
            order_type=order_type,
            limit_price=limit_price,
            stop_price=stop_price,
            order_id=f"DEMO-{self._order_counter:04d}",
            status=OrderStatus.PENDING,
            created_at=datetime.now(UTC),
        )
        self.pending_orders.append(order)
        return order

    async def cancel_order_async(self, order_id: str) -> bool:
        return False

    async def close_position_async(self, asset: str) -> Order | None:
        return None

## 1. Stale-Data Rejection

`SafeBroker` keeps a `MarketSnapshot` per asset whenever the engine receives bars. Every
order intent is checked against that snapshot's age and
`LiveRiskConfig.max_data_staleness_seconds`. Orders against a snapshot older than that
threshold raise `RiskLimitError`, because the freshest known price is no longer trustworthy
enough to size against.

The demo below sets a 1-second staleness threshold, accepts one order against a fresh
snapshot, then waits past the threshold and submits another order. The second order must
fail. There is no retry and no fallback to a stale price: the strategy waits for fresh data
or stands down.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 1. 拒絕過時資料

每當引擎收到價格長條，`SafeBroker` 都會為每項資產保留一份 `MarketSnapshot`。每項訂單意圖都會依快照的資料年齡與 `LiveRiskConfig.max_data_staleness_seconds` 檢查。若快照超過該門檻，就會觸發 `RiskLimitError`，因為最新已知價格已不可靠，不能用來計算部位規模。

下方示範將過時門檻設為 1 秒，先接受一筆使用新鮮快照的訂單，再等待超過門檻後送出另一筆訂單。第二筆訂單必須失敗。系統不會重試，也不會退回使用過時價格：策略須等待新資料，否則就停止動作。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
def _temp_state_path(prefix: str) -> Path:
    """Return a non-existent path inside the isolated notebook output directory."""
    fd, name = tempfile.mkstemp(prefix=prefix, suffix=".json", dir=STATE_DIR)
    os.close(fd)
    p = Path(name)
    p.unlink(missing_ok=True)
    return p


def _cleanup_state_path(path: Path) -> None:
    """Remove a temporary risk-state file and its default sibling journal."""
    journal = path.with_name(f"{path.stem}-journal{path.suffix or '.json'}l")
    for candidate in (
        path,
        path.with_name(f"{path.name}.lock"),
        journal,
        journal.with_name(f"{journal.name}.lock"),
        journal.with_name(f"{journal.name}.head"),
    ):
        candidate.unlink(missing_ok=True)


stale_state = _temp_state_path("nb13_stale_")
stale_broker = DemoBroker()
stale_safe = SafeBroker(
    stale_broker,
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_data_staleness_seconds=1.0,
        state_file=str(stale_state),
    ),
)
run_demo(stale_safe.connect())

# Fresh snapshot, order accepted
stale_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
fresh_order = run_demo(stale_safe.submit_order_async("DEMO", 10))
print(f"fresh_data_order: accepted ({fresh_order.order_type.value} {fresh_order.quantity} DEMO)")

# Wait past staleness threshold
time.sleep(STALE_DATA_WAIT_SECONDS)

try:
    run_demo(stale_safe.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"stale_data_block: {exc}")
else:
    raise AssertionError("Stale market data did not block the second order")

run_demo(stale_safe.disconnect())
_cleanup_state_path(stale_state)

fresh_data_order: accepted (market 10.0 DEMO)


stale_data_block: Stale market data for DEMO: 1.5s old exceeds max 1.0s


The second order never reaches the broker adapter, and the `RiskLimitError` carries both the
measured age and the configured threshold, which is what the operator needs to decide between
widening the staleness window, investigating the feed, and halting.

A feed that goes silent during a venue outage is a common live failure, and trading on the
last known price through the outage is how a stop-loss strategy turns into its own opposite.
The staleness check is the runtime enforcement of §25.7's pre-flight requirement that prices
be current rather than stale.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
第二筆訂單完全不會到達券商介接器，而 `RiskLimitError` 會同時包含測得的資料年齡與設定門檻，供操作人員判斷應擴大新鮮度視窗、調查資料來源，或停止交易。

平台故障時資料來源突然停止更新，是常見的實盤故障；若故障期間仍依最後已知價格交易，停損策略可能反而變成自身的反面。新鮮度檢查是在執行階段強制落實 §25.7 的啟動前要求：價格必須是最新的，而非過時的。
<!-- ML4T_ZH_END c0010-p000 -->


## 2. Auto Kill-Switch Trip on Daily-Loss Breach

Every order submission also reads the broker's account value. The first read anchors the
session's starting equity; each later one measures the intraday loss as the drop from that
anchor, floored at zero. When the loss exceeds `LiveRiskConfig.max_daily_loss`, the kill
switch latches and the order is rejected.

The demo simulates an equity drawdown by mutating the broker's `account_value` mid-session.
The first order anchors the session start; the equity drops; the next order trips the
kill switch. The persisted `RiskState` records the activation reason so a same-day restart
resumes with the latch intact.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 2. 超過每日損失限額時自動觸發 Kill Switch

每次送單也會讀取券商的帳戶價值。第一次讀取會建立工作階段起始權益基準；之後每次讀取則計算相對於該基準的盤中損失，最低為零。當損失超過 `LiveRiskConfig.max_daily_loss`，Kill Switch 就會鎖定並拒絕訂單。

示範會在工作階段中途變動券商的 `account_value`，模擬權益回撤。第一筆訂單建立工作階段起點；權益下跌後，下一筆訂單便觸發 Kill Switch。持久化的 `RiskState` 會記錄啟動原因，讓同一天重新啟動時仍維持鎖定。
<!-- ML4T_ZH_END c0011-p000 -->


In [6]:
killswitch_state = _temp_state_path("nb13_killswitch_")
killswitch_broker = DemoBroker(account_value=100_000.0)
killswitch_safe = SafeBroker(
    killswitch_broker,
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_daily_loss=KILL_SWITCH_LOSS_USD,
        max_data_staleness_seconds=60.0,
        state_file=str(killswitch_state),
    ),
)
run_demo(killswitch_safe.connect())
killswitch_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})

# Anchor session_start_equity at $100k
run_demo(killswitch_safe.submit_order_async("DEMO", 1))
print(f"pre_drawdown_state: kill_switch={killswitch_safe._state.kill_switch_activated}")

# Simulated equity drop
killswitch_broker.account_value = 100_000.0 - (KILL_SWITCH_LOSS_USD + 250.0)
killswitch_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 99.0}}, {})

try:
    run_demo(killswitch_safe.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"daily_loss_block: {exc}")
else:
    raise AssertionError("Daily-loss breach did not block the order")

print(
    "post_breach_state: "
    f"kill_switch={killswitch_safe._state.kill_switch_activated} "
    f"daily_loss=${killswitch_safe._state.daily_loss:,.0f} "
    f"reason={killswitch_safe._state.kill_switch_reason!r}"
)
run_demo(killswitch_safe.disconnect())

2026-09-08 21:35:57,115 CRITICAL ml4t.live.safety: KILL SWITCH ACTIVATED: Daily loss $1,250.00 exceeds max $1,000.00


pre_drawdown_state: kill_switch=False
daily_loss_block: Daily loss $1,250.00 exceeds max $1,000.00
post_breach_state: kill_switch=True daily_loss=$1,250 reason='Daily loss $1,250.00 exceeds max $1,000.00'


### 2a. Reconstructing `SafeBroker` from the State File

A real engine restart re-instantiates `SafeBroker` from its state file. The latch and the
activation reason persist; the next order is rejected before it reaches any risk check
downstream of the kill switch.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
### 2a. 從狀態檔重建 `SafeBroker`

真正的引擎重啟會從狀態檔重新建立 `SafeBroker` 執行個體。鎖定狀態與啟動原因都會保留；下一筆訂單在到達 Kill Switch 後續的任何風險檢查前，就會遭拒絕。
<!-- ML4T_ZH_END c0013-p000 -->


In [7]:
killswitch_safe2 = SafeBroker(
    DemoBroker(account_value=100_000.0),
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_daily_loss=KILL_SWITCH_LOSS_USD,
        max_data_staleness_seconds=60.0,
        state_file=str(killswitch_state),
    ),
)
print(
    "after_restart_state: "
    f"kill_switch={killswitch_safe2._state.kill_switch_activated} "
    f"reason={killswitch_safe2._state.kill_switch_reason!r}"
)
run_demo(killswitch_safe2.connect())
killswitch_safe2._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
try:
    run_demo(killswitch_safe2.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"post_restart_block: {exc}")
else:
    raise AssertionError("Persisted kill switch did not block the restarted broker")
run_demo(killswitch_safe2.disconnect())

# Reset for downstream demos
_cleanup_state_path(killswitch_state)

2026-09-08 21:35:57,184 WARNING ml4t.live.safety: Kill switch was previously activated: Daily loss $1,250.00 exceeds max $1,000.00


2026-09-08 21:35:57,185 WARNING ml4t.live.safety: SafeBroker reconciliation mismatch: missing_pending_orders=1


after_restart_state: kill_switch=True reason='Daily loss $1,250.00 exceeds max $1,000.00'
post_restart_block: Kill switch active: Daily loss $1,250.00 exceeds max $1,000.00


The kill switch enforces §25.7's requirement that it remain independent of the component it
controls. Because the latch lives in a state file, an engine crash and reconnect does not
silently re-enable trading; the operator must explicitly clear the latch before the engine
resumes. The Friday-after-hours pile-up that motivated the startup reconciliation in
`12_ib_basket_rebalance_demo` is the same failure in the position dimension rather than
the P&L dimension.

Latching the switch in persistent state is what makes it usable in an emergency. A
non-persistent kill switch is a flag in process memory, and the recovery from a disorderly
day is the worst time to discover the difference.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
Kill Switch 落實 §25.7 的要求：停止機制須獨立於其所控制的元件。由於鎖定狀態保存在狀態檔中，引擎當機並重新連線不會悄悄重新啟用交易；引擎恢復前，操作人員必須明確解除鎖定。`12_ib_basket_rebalance_demo` 啟動時對帳所針對的週五盤後訂單堆積，與此處的損益維度失敗相似，只是發生在部位維度。

將開關鎖定狀態持久化，才能在緊急情況下發揮作用。未持久化的 Kill Switch 只是處理程序記憶體中的旗標；在混亂的一天後進行復原，最不該在那時才發現兩者的差別。
<!-- ML4T_ZH_END c0015-p000 -->


## 3. Reconciliation Report on a Divergent State File

`SafeBroker.connect()` diffs the persisted snapshot from the previous session against the
broker's authoritative current state. The demo below writes a state file claiming the
strategy ended the previous session long 10 AAPL with a working LIMIT order, then connects
to a broker that knows about neither. The reconciliation report lists every divergence; a
production launcher refuses to start a new trading cycle until the report is clean or the
operator has explicitly cleared the persisted state.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 3. 狀態檔不一致時的對帳報告

`SafeBroker.connect()` 會比較上次工作階段保存的快照與券商目前的權威狀態。下方示範寫入狀態檔，聲稱策略在上次工作階段結束時持有 10 股 AAPL 多頭部位，且有一筆仍有效的 LIMIT 訂單；隨後連線至對兩者皆無紀錄的券商。對帳報告會列出所有差異；正式環境的啟動程序會拒絕開始新的交易循環，直到報告完全一致，或操作人員明確清除持久化狀態。
<!-- ML4T_ZH_END c0016-p000 -->


In [8]:
recon_state = _temp_state_path("nb13_recon_")
divergent = RiskState(
    date=datetime.now(UTC).date().isoformat(),
    persisted_positions={"AAPL": 10.0},
    persisted_pending_orders=[
        {
            "asset": "AAPL",
            "side": "buy",
            "quantity": 10.0,
            "order_type": "limit",
            "limit_price": 150.0,
        }
    ],
)
recon_state.write_text(json.dumps(divergent.to_dict(), indent=2))
recon_state.chmod(0o600)

recon_broker = DemoBroker(
    positions={
        "MSFT": Position(
            asset="MSFT",
            quantity=5,
            entry_price=410.0,
            entry_time=datetime.now(UTC),
        )
    }
)
recon_safe = SafeBroker(
    recon_broker,
    LiveRiskConfig(execution_mode="paper", state_file=str(recon_state)),
)
run_demo(recon_safe.connect())

report = recon_safe.reconciliation_report
print(f"clean: {report['clean']}")
print(f"missing_positions:    {report['missing_positions']}")
print(f"unexpected_positions: {report['unexpected_positions']}")
print(f"quantity_mismatches:  {report['quantity_mismatches']}")
print(f"missing_pending_orders:    {len(report['missing_pending_orders'])} order(s)")
print(f"unexpected_pending_orders: {len(report['unexpected_pending_orders'])} order(s)")
assert report["clean"] is False
assert report["missing_positions"] == {"AAPL": 10.0}
assert report["unexpected_positions"] == {"MSFT": 5.0}
assert len(report["missing_pending_orders"]) == 1

2026-09-08 21:35:57,292 WARNING ml4t.live.safety: SafeBroker reconciliation mismatch: missing_positions=['AAPL'], unexpected_positions=['MSFT'], missing_pending_orders=1


clean: False
missing_positions:    {'AAPL': 10.0}
unexpected_positions: {'MSFT': 5.0}
quantity_mismatches:  {}
missing_pending_orders:    1 order(s)
unexpected_pending_orders: 0 order(s)


The report names which positions and orders disagree, and in which direction. `missing_positions` are positions the persisted state expected and the broker no
longer reports, typically after a manual flatten, an after-hours fill, or an overnight
corporate action. `unexpected_positions` are positions present at the broker that the
persisted state did not know about, typically a fill that landed after the last persist,
or a position created out of band.

Reading the report is half the task; resolving it is the other half. The operator either
investigates the divergence in the broker GUI and re-runs once it is understood, or, when
the divergence is known to be benign, clears the persisted state and reconnects to a
clean baseline.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
報告會指出哪些部位與訂單不一致，以及差異的方向。`missing_positions` 是持久化狀態預期存在、券商卻已不再回報的部位，常見原因包括手動平倉、盤後成交或隔夜公司行動。`unexpected_positions` 是券商有紀錄、持久化狀態卻不知道的部位，常見原因包括上次保存後才完成的成交，或透過系統外途徑建立的部位。

讀懂報告只完成了一半工作；另一半是解決差異。操作人員可以在券商圖形介面調查差異，確認原因後重新執行；若已知差異無害，也可以清除持久化狀態，重新連線並建立一致的基準。
<!-- ML4T_ZH_END c0018-p000 -->


In [9]:
run_demo(recon_safe.disconnect())
_cleanup_state_path(recon_state)

# Clean baseline: persisted positions match the broker exactly
clean_state = _temp_state_path("nb13_recon_clean_")
matched_state = RiskState(
    date=datetime.now(UTC).date().isoformat(),
    persisted_positions={"MSFT": 5.0},
    persisted_pending_orders=[],
)
clean_state.write_text(json.dumps(matched_state.to_dict(), indent=2))
clean_state.chmod(0o600)

recon_broker_clean = DemoBroker(
    positions={
        "MSFT": Position(
            asset="MSFT",
            quantity=5,
            entry_price=410.0,
            entry_time=datetime.now(UTC),
        )
    }
)
recon_safe_clean = SafeBroker(
    recon_broker_clean,
    LiveRiskConfig(execution_mode="paper", state_file=str(clean_state)),
)
run_demo(recon_safe_clean.connect())
clean_report = recon_safe_clean.reconciliation_report
print(f"after_reset clean: {clean_report['clean']}")
assert clean_report["clean"] is True
run_demo(recon_safe_clean.disconnect())
_cleanup_state_path(clean_state)

after_reset clean: True


A production launcher should treat a non-clean reconciliation report the same way it treats
a failed authentication probe: refuse to launch and wait for
operator action. `12_ib_basket_rebalance_demo` implements that policy: the IB basket loop
will not submit if the report is not clean.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
正式環境的啟動程序對不一致的對帳報告，應採取與身分驗證探測失敗相同的處置：拒絕啟動，等待操作人員處理。`12_ib_basket_rebalance_demo` 實作了這項政策：若報告不一致，IB 籃子循環就不會送單。
<!-- ML4T_ZH_END c0020-p000 -->


## 4. Engine Health States

`LiveEngine.runtime_status()` returns a dict whose `health` field reduces engine and feed
state to a small set of operator-readable categories: `stopped`, `waiting_for_data`,
`ok`, `feed_silent`, `idle_market_closed`, and `broker_disconnected`. The demo below
constructs a tiny engine over a synthetic feed and a no-op strategy, runs it for a short
bounded window, halts the feed, and reports the health transitions. No real venue, no
broker connectivity, only the engine's own bookkeeping.

The toy feed declares no equity symbols, so the engine treats it as a continuous
market. This removes wall-clock session dependence and makes the health sequence
reproducible at any hour.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
## 4. 引擎健康狀態

`LiveEngine.runtime_status()` 回傳字典，其 `health` 欄位將引擎與資料來源狀態歸納為少數操作人員可理解的類別：`stopped`、`waiting_for_data`、`ok`、`feed_silent`、`idle_market_closed` 與 `broker_disconnected`。下方示範以合成資料來源及不執行動作的策略建立小型引擎，在短暫且有上限的時間視窗內執行，然後停止資料來源並報告健康狀態轉換。沒有真實平台，也沒有券商連線；只使用引擎自身的記錄。

玩具資料來源不宣告任何股票標的，因此引擎會將其視為連續市場。這排除了實際時間的交易時段依賴，使健康狀態序列在任何時段都可重現。
<!-- ML4T_ZH_END c0021-p000 -->


In [10]:
@dataclass
class IdleStrategy(Strategy):
    """Strategy that ignores every bar, so only the health state is under observation."""

    bars_seen: int = 0

    def __post_init__(self) -> None:
        try:
            super().__init__()
        except TypeError:
            pass

    def on_data(self, *args: Any, **kwargs: Any) -> None:
        self.bars_seen += 1

    def on_start(self, *args: Any, **kwargs: Any) -> None:
        return None

    def on_end(self, *args: Any, **kwargs: Any) -> None:
        return None

#### Toy market-data feed

A minimal async feed that emits a fixed number of bars then goes silent.
Pairing it with `IdleStrategy` lets the LiveEngine's health-state
transitions surface (`waiting_for_data` → `ok` → `feed_silent`) without
any real market dependency.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
#### 玩具市場資料來源

最簡單的非同步資料來源，送出固定數量的價格長條後便停止更新。將其與 `IdleStrategy` 配合，即可呈現 LiveEngine 的健康狀態轉換（`waiting_for_data` → `ok` → `feed_silent`），而不依賴任何真實市場。
<!-- ML4T_ZH_END c0023-p000 -->


In [11]:
@dataclass
class _ToyFeed:
    """Synthetic feed that emits one bar then goes silent until stopped."""

    initial_bars: int = 1
    start_delay_seconds: float = 0.2
    bar_interval_seconds: float = 0.1
    silent_after_initial: bool = True
    stopped: bool = field(default=False)

    async def start(self) -> None:
        self.stopped = False

    def stop(self) -> None:
        self.stopped = True

    def __aiter__(self):
        return self._stream()

    async def _stream(self):
        await asyncio.sleep(self.start_delay_seconds)
        for _ in range(self.initial_bars):
            if self.stopped:
                return
            yield (datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
            await asyncio.sleep(self.bar_interval_seconds)
        # Go silent: never yield more bars, just hold the iterator open
        while not self.stopped:
            await asyncio.sleep(0.1)

Wire the feed, strategy, and demo broker into a `LiveEngine` with a
short feed-silence timeout so the watchdog flips to `feed_silent`
quickly once the toy feed stops emitting bars.

<!-- ML4T_ZH_BEGIN c0025-p000 -->
將資料來源、策略與示範券商接入 `LiveEngine`，並設定較短的資料來源靜默逾時，讓玩具資料來源停止送出價格長條後，監看程序能迅速切換至 `feed_silent`。
<!-- ML4T_ZH_END c0025-p000 -->


In [12]:
broker_h = DemoBroker()
feed_h = _ToyFeed(initial_bars=2, start_delay_seconds=0.2, bar_interval_seconds=0.05)
strategy_h = IdleStrategy()
engine = LiveEngine(
    strategy=strategy_h,
    broker=broker_h,
    feed=feed_h,
    feed_silence_seconds=1.0,
    watchdog_poll_seconds=0.25,
)

`observe_engine` polls `runtime_status()` within a bounded window and records
each expected health transition when it occurs.

<!-- ML4T_ZH_BEGIN c0027-p000 -->
`observe_engine` 在有上限的時間視窗內輪詢 `runtime_status()`，並在每次預期的健康狀態轉換發生時記錄下來。
<!-- ML4T_ZH_END c0027-p000 -->


In [13]:
async def observe_engine() -> list[tuple[float, str]]:
    """Run the engine until each expected health transition occurs."""
    loop = asyncio.get_running_loop()
    started_at = loop.time()
    deadline = started_at + HEALTH_OBSERVATION_SECONDS
    transitions = [(0.0, engine.runtime_status()["health"])]

    async def wait_for_health(expected: str) -> None:
        while loop.time() < deadline:
            health = engine.runtime_status()["health"]
            if health == expected:
                transitions.append((round(loop.time() - started_at, 2), health))
                return
            await asyncio.sleep(0.01)
        observed = [health for _, health in transitions]
        raise AssertionError(f"expected health {expected!r}; observed {observed}")

    await engine.connect()
    run_task = asyncio.create_task(engine.run())
    for expected in ("waiting_for_data", "ok", "feed_silent"):
        await wait_for_health(expected)
    await engine.stop()
    run_task.cancel()
    try:
        await run_task
    except asyncio.CancelledError:
        pass
    transitions.append((round(loop.time() - started_at, 2), engine.runtime_status()["health"]))
    return transitions

In [14]:
timeline = run_demo(observe_engine())

In [15]:
last = None
transitions: list[dict] = []
for t, health in timeline:
    if health != last:
        transitions.append({"t_seconds": round(t, 2), "health": health})
        last = health
health_timeline = pl.DataFrame(transitions)
observed_health = health_timeline["health"].to_list()
assert observed_health == [
    "stopped",
    "waiting_for_data",
    "ok",
    "feed_silent",
    "stopped",
], observed_health
health_timeline

t_seconds,health
f64,str
0.0,"""stopped"""
0.01,"""waiting_for_data"""
0.21,"""ok"""
1.26,"""feed_silent"""
1.31,"""stopped"""


The printed sequence dedupes consecutive identical health states, so the observed output is `stopped → waiting_for_data → ok → feed_silent → stopped`.
A short initial delay makes the waiting state observable, and the feed's continuous-market
contract removes equity-session dependence. None of the transitions require a real venue.

`runtime_status()` is what a watchdog or supervisor process reads.
The categories are deliberately narrow because operators act on them under stress:
`feed_silent` calls for a different response than `broker_disconnected` even though both
look like "data stopped" from inside the strategy loop.

<!-- ML4T_ZH_BEGIN c0031-p000 -->
列印的序列會去除連續重複的健康狀態，因此觀察到的輸出是 `stopped → waiting_for_data → ok → feed_silent → stopped`。短暫的初始延遲讓等待狀態可見，而資料來源的連續市場契約排除了股票交易時段的依賴。所有轉換都不需要真實平台。

`runtime_status()` 是監看程序或主管程序讀取的資訊。這些類別刻意保持精簡，因為操作人員必須在壓力下據此行動：`feed_silent` 所需的回應與 `broker_disconnected` 不同，即使從策略循環內部看來，兩者都像是「資料停止」。
<!-- ML4T_ZH_END c0031-p000 -->


In [16]:
assert not list(STATE_DIR.iterdir()), list(STATE_DIR.iterdir())
STATE_DIR.rmdir()
assert not STATE_DIR.exists(), STATE_DIR
print("Temporary state artifacts: cleaned")

Temporary state artifacts: cleaned


## Key Takeaways

1. **Stale-data rejection** prevents trading on prices the runtime no longer trusts. The
   threshold is enforced per asset on every order intent.
2. **Daily-loss kill switch** latches in persisted state; reconstruction does not reset it,
   which is what makes it useful as an emergency control.
3. **Startup reconciliation** surfaces every divergence between the persisted snapshot and
   the broker's authoritative state. Production launchers refuse to start a new cycle
   against a non-clean report.
4. **Engine health states** reduce the runtime to a small operator-facing vocabulary that a
   supervisor process can consume directly.

**Next**: `10_safety_risk_demo` covers the configurable risk surface;
`12_ib_basket_rebalance_demo` shows the same controls in a live IB paper basket rebalance; §25.7 ties the abstract pre-flight requirements to the
enforced runtime behaviours demonstrated here.

<!-- ML4T_ZH_BEGIN c0033-p000 -->
## 重點整理

1. **拒絕過時資料**可防止使用執行階段不再信任的價格交易。每項訂單意圖都會按資產強制檢查門檻。
2. **每日損失 Kill Switch** 會鎖定在持久化狀態中；重建執行個體不會重設，這正是它能作為緊急控制措施的原因。
3. **啟動時對帳**會呈現持久化快照與券商權威狀態之間的每項差異。正式環境的啟動程序面對不一致的報告，會拒絕開始新循環。
4. **引擎健康狀態**將執行階段狀況歸納為少數面向操作人員的用語，供主管程序直接使用。

**接下來**：`10_safety_risk_demo` 涵蓋可設定的風險介面；`12_ib_basket_rebalance_demo` 示範相同控制措施如何用於即時 IB 模擬交易籃子再平衡；§25.7 則將抽象的啟動前要求與此處示範的執行階段強制行為連結起來。
<!-- ML4T_ZH_END c0033-p000 -->
